# 62. `ResonanceAmplitude` and `ResonanceContext` directly

**Objectives:**

- Build a resonance amplitude by hand from a lineshape, an angular factor, and a
  Blatt-Weisskopf barrier plugin, using `ResonanceAmplitude` directly -- bypassing
  the declarative `Resonance` dataclass.
- Construct the `ResonanceContext` (`parent_mass`, `daughter_masses`, `bachelor_mass`,
  `spin`, `pole_mass`, `pole_width`, `resonance_radius`, `parent_radius`) by hand.
- Confirm the manual amplitude numerically matches what `Resonance(...)` produces
  for the same inputs inside a `DecayModel`.

Run cells top to bottom in a fresh kernel. Masses are in GeV, invariants in GeV²,
daughter indices start at zero.

In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede any numerical work: amplitudes use complex128.

import jax.numpy as jnp
import numpy as np

from dalitzplotfitter import (
    CovariantAngular, DecayChannel, DecayModel, NonResonant, RealImag,
    RelativisticBreitWigner, Resonance, ResonanceAmplitude, ResonanceContext,
    generate_toy,
)

## 1. The declarative reference: `Resonance` inside a `DecayModel`

A K* on the `(K, pi+)` pair of `B+ -> K+ pi+ pi-`, all three daughters distinct so
there is no identical-particle symmetrization to reason about. `DecayModel` builds a
`ResonanceAmplitude` for this internally (`DecayModel._build_resonance`), from the
same inputs we pass here.

In [2]:
channel = DecayChannel("B+", ("K+", "pi+", "pi-"))
declarative_model = DecayModel(
    channel,
    [Resonance("Kstar", (0, 1), RealImag(1.0, 0.0), mass=0.8958, width=0.0474, spin=1),
     NonResonant(RealImag(0.3, -0.1), name="NR")],
    normalization_method="square-dalitz", normalization_pair=(0, 1),
    normalization_resolution=60,
)
truth = {p.name: p.value for p in declarative_model.parameters}
data = generate_toy(
    declarative_model, 800, parameters=truth, seed=62,
    method="inverse-transform", inverse_resolution=256, include_momenta=False,
)
declarative_kstar = None
for component in declarative_model.amplitude_model.components:
    if component.name == "Kstar":
        declarative_kstar = component.function
print("Found the declarative Kstar dynamics:", declarative_kstar is not None)

Found the declarative Kstar dynamics: True


## 2. Build the same amplitude by hand

`ResonanceContext` carries the kinematic/particle-property inputs a lineshape and
barrier need: `parent_mass`, the two `daughter_masses` of the resonance's own pair
(`(K, pi+)` here), the `bachelor_mass` (`pi-`, index 2, the daughter *not* in the
pair), `spin`, `pole_mass`, `pole_width`, and the two barrier radii
`resonance_radius`/`parent_radius` (GeV^-1, Laura++ convention, defaults 1.5/5.0).

`ResonanceAmplitude` then composes a `lineshape(mass, context)`, an `angular` factor
and the Blatt-Weisskopf barriers through `daughter_key`/`partner_key`/`bachelor_key`
-- `"p1"`/`"p2"`/`"p3"` map to the channel's daughter indices 0/1/2, in the same
order `DecayModel._build_resonance` uses: `daughter_key` and `partner_key` are the
pair, `bachelor_key` is the remaining index.

In [3]:
context = ResonanceContext(
    parent_mass=channel.parent_mass,
    daughter_masses=(channel.daughter_masses[0], channel.daughter_masses[1]),
    bachelor_mass=channel.daughter_masses[2],
    spin=1,
    pole_mass=0.8958,
    pole_width=0.0474,
    resonance_radius=1.5,
    parent_radius=5.0,
)
manual_kstar = ResonanceAmplitude(
    context=context,
    daughter_key="p1", partner_key="p2", bachelor_key="p3",
    lineshape=RelativisticBreitWigner(),
    angular=CovariantAngular(),
)
print(context)

ResonanceContext(parent_mass=5.2794099999999995, daughter_masses=(0.49367700000000003, 0.13957039000000002), bachelor_mass=0.13957039000000002, spin=1, pole_mass=0.8958, pole_width=0.0474, resonance_radius=1.5, parent_radius=5.0)


## 3. Compare on the toy data

Both dynamics functions accept the same event-data mapping (`supports_invariant_input`
lets `ResonanceAmplitude` work straight from `s12`/`s13`/`s23`, no four-momenta
needed) and no fit parameters here, since every input above is a plain float.

In [4]:
data_dict = data.as_dict()
declarative_values = jnp.asarray(declarative_kstar(data_dict, None))
manual_values = jnp.asarray(manual_kstar(data_dict, None))

np.testing.assert_allclose(manual_values, declarative_values, atol=1e-10)
print("Manual ResonanceAmplitude matches the Resonance(...)-built dynamics exactly.")
print("Sample values:", np.asarray(manual_values[:3]))

Manual ResonanceAmplitude matches the Resonance(...)-built dynamics exactly.
Sample values: [ 24.26342195-71.62272125j  16.97875826-74.97724248j
 -43.70627215-80.85145369j]


## 4. What `Resonance(...)` adds on top

`ResonanceAmplitude` is the plugin-composition engine; `Resonance` is the thin
declarative wrapper `DecayModel._build_resonance` uses to build exactly the
`ResonanceContext`/`ResonanceAmplitude` pair assembled by hand above, from a mass,
width, spin, pair and (optionally) mass/width looked up from `particle` by name.
Coefficients, `normalize_component` and identical-particle symmetrization
(`final_state`) are handled by `Resonance`/`DecayModel`, not by `ResonanceAmplitude`
itself -- see [`docs/dynamics_structure.md`](../../docs/dynamics_structure.md). A
`ResonanceAmplitude` built by hand, like `manual_kstar` above, still evaluates
correctly on raw event data; it is only that this repository's own dataclass
plugin-resolution machinery (`resolve_value`, used to substitute floating fit
`Parameter`s into dataclass fields such as `ResonanceContext`) is not meant to walk
into a *second* `ResonanceAmplitude`/`ResonanceContext` nested as another
component's `dynamics` -- CLAUDE.md's "Two API layers" and "One-dimensional
lineshapes vs. full 2D Dalitz amplitudes" sections describe `Resonance` and
`DalitzAmplitude`/`QMI2D` as the two supported ways to add a component to a
`DecayModel`; `ResonanceAmplitude` itself is a public building block for tests and
diagnostics like this one, called directly rather than re-wrapped.

## Continue learning

See [`docs/dynamics_structure.md`](../../docs/dynamics_structure.md) for the
lineshape/angular/barrier plugin interfaces `ResonanceAmplitude` composes, and
CLAUDE.md's "One-dimensional lineshapes vs. full 2D Dalitz amplitudes" section for
when a resonance instead needs `DalitzAmplitude`/`QMI2D`
(see [tutorial 63](tutorial_63_dalitz_amplitude_qmi2d.ipynb)).

Return to [the course guide](TUTORIALS.md).